In [ ]:
import base64, hashlib, importlib.util, json, os, re, shutil, signal, subprocess, sys
import threading, time, types, urllib.error, urllib.request, zipfile, zlib
from datetime import datetime, timezone
from pathlib import Path

NOTEBOOK_START = time.time()

# ---------- 可以改的设置 ----------
# 不设时间限制，每一组都跑完。断线或手动停止后，重新运行全部单元格即可：
# 已经跑完的组直接跳过（不再下载模型），没跑完的组从断点接着跑。
RUN_COT = True             # 四个模型的 basic 跑完后，再跑 cot 模板的前 200 条
OLLAMA_VERSION = "0.40.3"  # 与 Qwen 各组相同
CONTEXT_LENGTH = 8192      # 与 Qwen 各组相同
REQUEST_TIMEOUT = 600
# 每个系列按顺序尝试，第一个能下载的标签就用它；系列按模型大小从小到大排，先跑快的
FAMILIES = {
    "Mistral": ["mistral:7b", "mistral"],
    "Llama": ["llama3.1:8b", "llama3:8b"],
    "Gemma": ["gemma3:12b", "gemma3:4b", "gemma2:9b"],
    "Phi-4": ["phi4:14b", "phi4"],
}

try:
    ON_COLAB = "COLAB_RELEASE_TAG" in os.environ or importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    ON_COLAB = False
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/sst2_more_models")  # 结果放 Drive，断开后重跑会接着跑
    MODEL_DIR = Path("/content/ollama_models")               # 模型权重放本机磁盘
elif Path("/kaggle/working").is_dir():
    ROOT, MODEL_DIR = Path("/kaggle/working/sst2_more_models"), Path("/tmp/ollama_models")
else:  # 腾讯 Cloud Studio 等其他 Linux 机器：结果放在当前目录下
    ROOT, MODEL_DIR = Path.cwd() / "sst2_more_models", Path.home() / "ollama_models"
DATA_DIR, META_DIR, RESULTS = ROOT / "data", ROOT / "metadata", ROOT / "ollama_results"
for directory in (ROOT, DATA_DIR, META_DIR, RESULTS, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# 换了平台时，把上一次的结果包放在 sst2_more_models 文件夹旁边（Kaggle 上可以作为 Dataset 挂上），
# 这里会先把它解压进来，已经跑完的组就会跳过，没跑完的组接着跑。
PREVIOUS_ZIP = "sst2_more_models_results.zip"
if not any(RESULTS.rglob("*.jsonl")):
    candidates = [ROOT.parent / PREVIOUS_ZIP, Path.cwd() / PREVIOUS_ZIP, Path.home() / PREVIOUS_ZIP]
    candidates += sorted(Path("/kaggle/input").rglob(PREVIOUS_ZIP)) if Path("/kaggle/input").is_dir() else []
    previous = next((c for c in candidates if c.is_file()), None)
    if previous:
        with zipfile.ZipFile(previous) as archive:
            names = [n for n in archive.namelist() if n.startswith("sst2_more_models/") and ".." not in n
                     and not n.endswith("/")]
            for name in names:
                target = ROOT / name.removeprefix("sst2_more_models/")
                if not target.exists():
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.write_bytes(archive.read(name))
        print(f"已从 {previous} 导入上一次的结果（{len(names)} 个文件）")
    else:
        print("没有找到上一次的结果包，所有组从头跑")
EVAL_SCRIPT = ROOT / "sst2_instruction_eval.py"
DATA = DATA_DIR / "sst2_validation.tsv"
HOST = "http://127.0.0.1:11434"  # 与 Qwen 各组相同（实验指纹里含这个地址）


def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


NOTES_PATH = META_DIR / "notes.json"
NOTES = json.loads(NOTES_PATH.read_text(encoding="utf-8")) if NOTES_PATH.exists() else []  # 保留以前几次运行的记录
SESSION = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


def note(message):
    print(message, flush=True)
    NOTES.append({"session": SESSION, "minute": round((time.time() - NOTEBOOK_START) / 60, 1), "message": message})
    save_json(NOTES_PATH, NOTES)


print("结果目录：", ROOT)
print("计划：", "、".join(FAMILIES), "各跑 basic 全部 872 条" + ("，再跑 cot 前 200 条" if RUN_COT else "")
      + "；不设时间限制，已跑完的组跳过")

In [ ]:
EMBEDDED_ASSETS = {
 "sst2_instruction_eval.py": {
  "sha256": "c3c96be1fe1348be4a8075894bca3ee07a9cf1458048b260b4b700613d875a95",
  "bytes": 43601,
  "zlib_base64": "..."
 },
 "sst2_validation.tsv": {
  "sha256": "a7a3bfee7fc80566578791dfea290715934cc82f436c893361c4d0108a3554f8",
  "bytes": 99186,
  "zlib_base64": "..."
 },
 "sst2_validation.metadata.json": {
  "sha256": "d9c018c30511f87199684de5b848a8037e968412ec101db82677663228a4e1ef",
  "bytes": 566,
  "zlib_base64": "eNptUstugzAQvEfKP0RcW8L6gTF8R+/IGBssOTbCkLaK8u811JWI1Jtndmdn1trH+XS5ZL1YRFBL1lwy973mN2uLwa4qe9+r0jtthq0YwoITGSZrdsFdWBP1xrtUcZHlFf4FVnTKttKvbgmRf2xkpFF8U0rfE4QNYr6h50F3VMAeTg3R6Z6CpTnZ5IPZ2YNexVjrnqr9d1SOjiPQQTnqVhs3qHmajds3BIU7ACCko5zVRP/9wChwybYGUQnSaaUqLTmUjJUVr2rUayVwDRUqa0Kl5FhTwiSvCWFI0h4QcEHKkmqeBvb+01kv+la5fvLJfFyWKTRFMer8ZubZz1fpb0mgviY/L6pvxd6KAbMcQY7wByINkIbyK2YVovAG0AAkldfaSCNsG/w6S/Visg5D3F0LqaJNkc4iFC9HcT49fwDBxJUo"
 }
}

DESTINATIONS = {"sst2_instruction_eval.py": EVAL_SCRIPT, "sst2_validation.tsv": DATA,
                "sst2_validation.metadata.json": DATA_DIR / "sst2_validation.metadata.json"}
for name, item in EMBEDDED_ASSETS.items():
    raw = zlib.decompress(base64.b64decode(item["zlib_base64"]))
    if len(raw) != item["bytes"] or hashlib.sha256(raw).hexdigest() != item["sha256"]:
        raise RuntimeError(f"内置文件 {name} 校验失败。")
    DESTINATIONS[name].write_bytes(raw)
save_json(META_DIR / "embedded_assets.json",
          {name: {k: v for k, v in item.items() if k != "zlib_base64"} for name, item in EMBEDDED_ASSETS.items()})
_spec = importlib.util.spec_from_file_location("sst2_instruction_eval", EVAL_SCRIPT)
EV = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(EV)
ROWS = EV.load_rows(types.SimpleNamespace(data_file=str(DATA), n=0))[0]
if len(ROWS) != 872 or sum(r["label"] == 1 for r in ROWS) != 444:
    raise RuntimeError("验证集条数或标签数量不对。")
print("评测脚本与数据已就绪：872 条，正面 444 条")

In [ ]:
if not shutil.which("nvidia-smi") or subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise RuntimeError("没有 GPU。Colab：代码执行程序 → 更改运行时类型 → T4 GPU；Cloud Studio：用“高性能”的 GPU 工作空间。")
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
(META_DIR / "nvidia_smi_start.txt").write_text(smi, encoding="utf-8")
print(smi.split("\n")[8] if len(smi.split("\n")) > 8 else smi)

LOCAL_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))
ENV = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_MODELS=str(MODEL_DIR), OLLAMA_NUM_PARALLEL="1",
           OLLAMA_CONTEXT_LENGTH=str(CONTEXT_LENGTH), OLLAMA_KEEP_ALIVE="30m", OLLAMA_MAX_LOADED_MODELS="1",
           OLLAMA_NOPRUNE="1")


def api(endpoint, payload=None, timeout=10):
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(HOST + endpoint, data=body, headers={"Content-Type": "application/json"})
    with LOCAL_OPENER.open(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def version_matches(text):
    return re.search(r"(?<![0-9.])" + re.escape(OLLAMA_VERSION) + r"(?![0-9.])", text or "") is not None


installed = ""
if shutil.which("ollama"):
    installed = subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout
if not version_matches(installed):
    installer = Path("/tmp/ollama_install.sh")
    try:
        missing = [name for name in ("curl", "zstd") if not shutil.which(name)]  # 安装脚本要用
        if missing:
            apt_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
            subprocess.run(["apt-get", "update", "-qq"], env=apt_env)
            subprocess.run(["apt-get", "install", "-y", "-qq"] + missing, env=apt_env, check=True)
        subprocess.run(["curl", "-fSL", "--retry", "3", "--connect-timeout", "30", "https://ollama.com/install.sh",
                        "-o", str(installer)], check=True)
        subprocess.run(["sh", str(installer)], env=dict(os.environ, OLLAMA_VERSION=OLLAMA_VERSION), check=True)
    except subprocess.CalledProcessError:
        if not installed:
            raise RuntimeError("下载 Ollama 失败，而且这台机器上也没有 Ollama。可以换用带 Ollama 的模板创建工作空间。")
        print(f"安装 Ollama {OLLAMA_VERSION} 失败，改用机器上已有的版本：{installed.strip()}")

subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)  # 同一会话重跑时关掉旧服务
time.sleep(2)
server_log = (META_DIR / "ollama_server.log").open("a", encoding="utf-8")
SERVER = subprocess.Popen(["ollama", "serve"], env=ENV, stdout=server_log, stderr=subprocess.STDOUT,
                          start_new_session=True)
deadline = time.monotonic() + 120
while True:
    if SERVER.poll() is not None:
        raise RuntimeError("Ollama 服务启动失败，见 metadata/ollama_server.log")
    try:
        version = api("/api/version", timeout=2).get("version")
        break
    except (urllib.error.URLError, TimeoutError, OSError):
        if time.monotonic() > deadline:
            raise RuntimeError("等待 Ollama 服务超时")
        time.sleep(1)
if version != OLLAMA_VERSION:
    note(f"Ollama 版本是 {version}，不是之前用的 {OLLAMA_VERSION}（版本记在 metadata 里）")
print("Ollama", version, "已就绪")

In [ ]:
MODELS = {}  # 系列 -> (标签, digest)


def safe(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "-", text)


def group_files(family, prompt, n, suffix):
    """这一组以前留下的文件（任一候选标签、任一模型版本）。"""
    size = "all" if n == 0 else f"n{n}"
    return [path for tag in FAMILIES[family]
            for path in RESULTS.glob(f"{safe(tag)}_*/ollama_{safe(tag)}_{prompt}_think-off_{size}_*{suffix}")]


def finished(family, prompt, n):
    for path in group_files(family, prompt, n, "_summary.json"):
        s = json.loads(path.read_text(encoding="utf-8"))
        if s.get("completed") and s.get("n") == (872 if n == 0 else n):
            return True
    return False


def rows_done(family, prompt, n):
    return max((sum(1 for line in path.read_text(encoding="utf-8").splitlines() if line.strip())
                for path in group_files(family, prompt, n, ".jsonl")), default=0)


def pull(family):
    for tag in FAMILIES[family]:
        started = time.monotonic()
        result = subprocess.run(["ollama", "pull", tag], env=ENV, capture_output=True, text=True, timeout=3600)
        if result.returncode != 0:
            note(f"{tag} 下载失败：{' '.join((result.stdout + result.stderr).split())[-300:]}")
            continue
        entry = next((m for m in api("/api/tags").get("models", []) if tag in (m.get("name"), m.get("model"))), None)
        if not entry or not entry.get("digest"):
            note(f"{tag} 下载后找不到 digest")
            continue
        digest = entry["digest"].removeprefix("sha256:")
        earlier = sorted(p.name.removeprefix(safe(tag) + "_") for p in RESULTS.glob(f"{safe(tag)}_*") if p.is_dir())
        meta_path = META_DIR / f"model_{safe(tag)}.json"
        if earlier and digest[:12] not in earlier:  # 模型版本变了：没跑完的组只能从头跑，旧的模型信息另存
            note(f"{tag} 的模型版本和上次不同（上次 {'、'.join(earlier)}，这次 {digest[:12]}），这个模型没跑完的组从头跑")
            meta_path = META_DIR / f"model_{safe(tag)}_{digest[:12]}.json"
        save_json(meta_path,
                  {"family": family, "tag": tag, "digest": digest, "tag_entry": entry,
                   "show": api("/api/show", {"model": tag}, timeout=60),
                   "pull_seconds": round(time.monotonic() - started), "ollama_version": version})
        note(f"{family}：用 {tag}（{entry.get('details', {}).get('parameter_size', '?')}，"
             f"{entry.get('details', {}).get('quantization_level', '?')}），下载 {time.monotonic() - started:.0f} 秒")
        return tag, digest
    return None, None


def run_group(tag, digest, prompt, n):
    out_dir = RESULTS / f"{safe(tag)}_{digest[:12]}"
    out_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(EVAL_SCRIPT), "--provider", "ollama", "--base-url", HOST, "--model", tag,
               "--prompt", prompt, "--think", "off", "--data-file", str(DATA), "--workers", "1",
               "--timeout", str(REQUEST_TIMEOUT), "--out-dir", str(out_dir), "--n", str(n)]
    label = f"{tag} {prompt} {'全部' if n == 0 else f'前 {n} 条'}"
    log_path = out_dir / f"{prompt}_n{n}_console.log"
    process = subprocess.Popen(command, cwd=ROOT, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", bufsize=1, start_new_session=True)
    started = time.monotonic()
    try:
        with log_path.open("a", encoding="utf-8") as log:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
        process.wait()
    except KeyboardInterrupt:  # 手动停止：让评测脚本保存断点后退出，下次接着跑
        os.killpg(process.pid, signal.SIGINT)
        process.wait(timeout=60)
        note(f"{label}：手动停止，已完成的条目存在断点里")
        raise
    note(f"{label}：结束（返回码 {process.returncode}），用时 {(time.monotonic() - started) / 60:.1f} 分钟")


PLAN = [(family, "basic", 0) for family in FAMILIES] + ([(family, "cot", 200) for family in FAMILIES] if RUN_COT else [])
todo = []
for family, prompt, n in PLAN:
    target = 872 if n == 0 else n
    if finished(family, prompt, n):
        print(f"{family} {prompt}：已跑完 {target} 条，跳过")
    else:
        done = rows_done(family, prompt, n)
        print(f"{family} {prompt}：" + (f"已有 {done}/{target} 条，接着跑" if done else f"要跑 {target} 条"))
        todo.append((family, prompt, n))
for family, prompt, n in todo:
    if family not in MODELS:
        MODELS[family] = pull(family)
    tag, digest = MODELS[family]
    if tag is None:
        note(f"{family} 的所有候选标签都下载失败，跳过")
        continue
    run_group(tag, digest, prompt, n)
print("全部组都处理完了" if todo else "所有组都已经跑完，直接运行下一个单元格打包结果")

In [ ]:
# 准确率按当前的解析规则，从每条的原始回答重新判分（以前跑的条目也一样）
RUN = re.compile(r"ollama_(.+)_(basic|cot)_think-off_(all|n\d+)_[0-9a-f]{12}\.jsonl$")
print(f"{'模型':<16}{'模板':<7}{'已跑/目标':>10}{'正确':>6}{'准确率':>9}{'无效':>5}{'截断':>5}{'秒/条':>7}")
for path in sorted(RESULTS.rglob("ollama_*.jsonl")):
    m = RUN.search(path.name)
    if not m:
        continue
    records = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line)
            if r.get("error") is None:
                records[r["idx"]] = r
    recs = list(records.values())
    target = 872 if m.group(3) == "all" else int(m.group(3)[1:])
    preds = [EV.parse_label(r["content"])[0] for r in recs]
    correct = sum(p == r["label"] for p, r in zip(preds, recs))
    acc_text = f"{correct / len(recs):.2%}" if recs else "—"
    avg_text = f"{sum(r['seconds'] for r in recs) / len(recs):.2f}" if recs else "—"
    print(f"{m.group(1):<15} {m.group(2):<7}{f'{len(recs)}/{target}':>10}{correct:>6}{acc_text:>9}"
          f"{sum(p is None for p in preds):>5}{sum(r.get('finish') == 'length' for r in recs):>5}{avg_text:>7}")

try:
    (META_DIR / "nvidia_smi_end.txt").write_text(
        subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout, encoding="utf-8")
except OSError as e:
    print("记录 nvidia-smi 失败：", e)
ZIP_PATH = ROOT.parent / "sst2_more_models_results.zip"
allowed = {".json", ".jsonl", ".csv", ".tsv", ".txt", ".log", ".py"}
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(p for p in ROOT.rglob("*") if p.is_file() and p.suffix.lower() in allowed):
        archive.write(path, arcname="sst2_more_models/" + path.relative_to(ROOT).as_posix())
where = (" Google Drive 的 MyDrive 文件夹里" if ON_COLAB else "右侧 Output 面板里" if Path("/kaggle/working").is_dir()
         else f"左侧文件列表的 {ZIP_PATH.parent} 下")
print(f"结果包已保存：{ZIP_PATH}（{ZIP_PATH.stat().st_size / 2 ** 20:.1f} MB）。"
      f"还需要你做的：在{where}找到它，下载后发给 Claude。")